In [ ]:
from pathlib import Path
from open_video_summary.demo import prepare_demo

from open_video_summary.utils.config import DataPaths, PROJECT_DIR
from open_video_summary.parsers.video import VideoLoader, VideoDumper
from open_video_summary.core.segmenter.video_segmenter import WordVideoSegmenter

# Unzipping sample dataset video files

In [ ]:
# Extract sample videos without replacing the versioned segmentation JSON.
prepare_demo(download_model=False)


# Loading data, segmenting videos, and saving output file

In [ ]:

# Only this dataset includes raw MP4s in the clone.
# Add other datasets after downloading their source videos.
datasets = ["bebe_real"]
for dataset in datasets:
    dataset_videos_path = (Path(DataPaths.RAW_PATH) / dataset).as_posix()
    dataset_json_path = (PROJECT_DIR / "outputs" / f"{dataset}_segments.json").as_posix()

    video_segmenter = WordVideoSegmenter(
        # Use "medium" for the original larger transcription model.
        whisper_model="base",
        min_segment_length=5,
        max_segment_length=120,
    )

    videos = [
        raw_video for raw_video in VideoLoader.load_videos_from_directory(dataset_videos_path)
    ]

    videos = video_segmenter.create_videos_segments(videos)
    VideoDumper.dump_videos_to_json(videos, dataset_json_path)

# Dumping segments to CSV file for analysis

In [ ]:

# Only this dataset includes raw MP4s in the clone.
# Add other datasets after downloading their source videos.
datasets = ["bebe_real"]

data = []
for dataset in datasets:
    dataset_videos_path = (Path(DataPaths.RAW_PATH) / dataset).as_posix()
    dataset_json_path = (PROJECT_DIR / "outputs" / f"{dataset}_segments.json").as_posix()

    videos = VideoLoader.load_videos_from_json(dataset_json_path)
    for video in videos:
        for i, seg in enumerate(video.segments):
            data.append((dataset, video.name, seg.content, seg.start, seg.end))

In [ ]:
import pandas as pd

df = pd.DataFrame(data, columns=["dataset", "video_name", "segment", "start_time", "end_time"])
df.start_time = df.start_time.astype(str).replace("\.", ",", regex=True)
df.end_time = df.end_time.astype(str).replace("\.", ",", regex=True)
df

In [ ]:
output_csv = PROJECT_DIR / "outputs/segments.csv"
output_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_csv, encoding="latin1", index=False)